In [15]:
import os
from typing import Tuple

import numpy as np
import pandas as pd


def speed_kmh_from_dist_time(dist_km, time_min):
    if pd.isna(dist_km) or pd.isna(time_min) or time_min <= 0:
        return np.nan
    return 60.0 * dist_km / time_min


def iqr_outlier_flags(series: pd.Series, k: float = 1.5) -> pd.Series:
    s = series.dropna()

    if s.empty:
        return pd.Series(False, index=series.index)

    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)

    iqr = q3 - q1

    lo = q1 - k * iqr
    hi = q3 + k * iqr

    return (series < lo) | (series > hi)


def run_checks(
    od_csv=r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\od_matrix_medium.csv",
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if not os.path.exists(od_csv):
        raise FileNotFoundError(f"Datei nicht gefunden: {od_csv}")

    od = pd.read_csv(od_csv)

    need_cols = {
        "from",
        "to",
        "profile",
        "load_state",
        "dist_km",
        "time_min",
        "cost",
        "reachable",
        "path_len",
        "tunnel_used",
    }

    missing = [c for c in need_cols if c not in od.columns]

    if missing:
        raise ValueError(f"Spalten fehlen: {missing}")

    rep = od.copy()

    # Vollständigkeit

    n_points = int(pd.unique(rep[["from", "to"]].values.ravel("K")).size)

    n_profiles = rep["profile"].nunique()
    n_load = rep["load_state"].nunique()

    expected_rows_est = n_points * (n_points - 1) * n_profiles * n_load

    # Grundchecks

    rep["diag_violation"] = rep["from"] == rep["to"]

    rep["neg_dist"] = (
        (rep["reachable"] == True)
        & (rep["dist_km"] < 0)
    )

    rep["neg_time"] = (
        (rep["reachable"] == True)
        & (rep["time_min"] < 0)
    )

    rep["neg_cost"] = (
        (rep["reachable"] == True)
        & (rep["cost"] < 0)
    )

    rep["cost_inf"] = np.isinf(rep["cost"])
    rep["cost_nan"] = rep["cost"].isna()

    # Distanz / Zeit

    rep["min_per_km"] = rep.apply(
        lambda r:
        (r["time_min"] / r["dist_km"])
        if (
            r["reachable"]
            and pd.notna(r["dist_km"])
            and r["dist_km"] > 0
        )
        else np.nan,
        axis=1,
    )

    rep["min_per_km_outside_gross_band"] = (
        (rep["min_per_km"] < 0.75)
        | (rep["min_per_km"] > 3.0)
    )

    rep["pathlen_violation"] = (
        (rep["reachable"] == True)
        & (rep["path_len"] < 2)
    )

    rep["v_avg_kmh"] = rep.apply(
        lambda r:
        speed_kmh_from_dist_time(
            r["dist_km"],
            r["time_min"]
        )
        if r["reachable"]
        else np.nan,
        axis=1,
    )

    rep["v_avg_outside_bounds"] = (
        (rep["v_avg_kmh"] < 10)
        | (rep["v_avg_kmh"] > 120)
    )

    # Profilchecks

    def profile_monotonic_flags(df_pair):

        sub = df_pair.set_index("profile")

        def get(col, prof):
            return sub[col].get(prof, np.nan)

        d_short = get("dist_km", "shortest")
        d_fast = get("dist_km", "fastest")
        d_safe = get("dist_km", "safest")

        t_short = get("time_min", "shortest")
        t_fast = get("time_min", "fastest")
        t_safe = get("time_min", "safest")

        return pd.Series(
            {
                "dist_violate_short_fast":
                    (
                        pd.notna(d_short)
                        and pd.notna(d_fast)
                        and d_short > d_fast + 1e-6
                    ),

                "time_violate_fast_vs_short":
                    (
                        pd.notna(t_short)
                        and pd.notna(t_fast)
                        and t_fast > t_short + 1e-6
                    ),

                "safe_dominates_fastest":
                    (
                        pd.notna(d_safe)
                        and pd.notna(d_fast)
                        and pd.notna(t_safe)
                        and pd.notna(t_fast)
                        and d_safe < d_fast - 1e-6
                        and t_safe < t_fast - 1e-6
                    ),
            }
        )

    prof_flags = (
        rep.groupby(
            ["from", "to", "load_state"]
        )
        .apply(profile_monotonic_flags)
        .reset_index()
    )

    rep = rep.merge(
        prof_flags,
        on=["from", "to", "load_state"],
        how="left",
    )

    # Profilidentität

    def identical_profile_check(df_pair):

        sub = df_pair.set_index("profile")

        required = {"shortest", "fastest", "safest"}

        if not required.issubset(set(sub.index)):
            return pd.Series(
                {"all_profiles_identical": False}
            )

        d = sub["dist_km"]
        t = sub["time_min"]

        same = (
            np.isclose(d["shortest"], d["fastest"])
            and np.isclose(d["shortest"], d["safest"])
            and np.isclose(t["shortest"], t["fastest"])
            and np.isclose(t["shortest"], t["safest"])
        )

        return pd.Series(
            {"all_profiles_identical": same}
        )

    ident_flags = (
        rep.groupby(
            ["from", "to", "load_state"]
        )
        .apply(identical_profile_check)
        .reset_index()
    )

    rep = rep.merge(
        ident_flags,
        on=["from", "to", "load_state"],
        how="left",
    )

    # Tunnelverbot

    rep["tunnel_used_violation_when_loaded"] = (
        (rep["load_state"] == "loaded")
        & (rep["tunnel_used"] == True)
    )

    # Symmetrie

    rep_key = rep.set_index(
        ["from", "to", "profile", "load_state"]
    )

    pairs = []

    for (a, b, prof, ls), row in rep_key.iterrows():

        rev_key = (b, a, prof, ls)

        if rev_key not in rep_key.index:
            continue

        rb = rep_key.loc[rev_key]

        da = row["dist_km"]
        db = rb["dist_km"]

        if (
            pd.notna(da)
            and pd.notna(db)
            and da >= 0
            and db >= 0
        ):

            mean = (da + db) / 2

            asym = (
                abs(da - db) / mean
                if mean > 0
                else np.nan
            )

        else:
            asym = np.nan

        pairs.append(
            (a, b, prof, ls, asym)
        )

    asym_df = pd.DataFrame(
        pairs,
        columns=[
            "from",
            "to",
            "profile",
            "load_state",
            "dist_asymmetry",
        ],
    )

    rep = rep.merge(
        asym_df,
        on=["from", "to", "profile", "load_state"],
        how="left",
    )

    rep["dist_asym_gt_0_2"] = (
        rep["dist_asymmetry"] > 0.2
    )

    # IQR-Ausreißer

    rep["dist_outlier_iqr"] = (
        rep.groupby(
            ["profile", "load_state"]
        )["dist_km"]
        .transform(
            lambda s: iqr_outlier_flags(s)
        )
    )

    rep["time_outlier_iqr"] = (
        rep.groupby(
            ["profile", "load_state"]
        )["time_min"]
        .transform(
            lambda s: iqr_outlier_flags(s)
        )
    )

    # Summary

    summary = {
        "rows_found": len(rep),
        "rows_expected_theoretical_estimate":
            int(expected_rows_est),

        "diag_violations":
            int(rep["diag_violation"].sum()),

        "negative_dist":
            int(rep["neg_dist"].sum()),

        "negative_time":
            int(rep["neg_time"].sum()),

        "negative_cost":
            int(rep["neg_cost"].sum()),

        "cost_inf":
            int(rep["cost_inf"].sum()),

        "cost_nan":
            int(rep["cost_nan"].sum()),

        "min_per_km_outside_gross_band":
            int(rep["min_per_km_outside_gross_band"].sum()),

        "pathlen_violations":
            int(rep["pathlen_violation"].sum()),

        "v_avg_outside_bounds":
            int(rep["v_avg_outside_bounds"].sum()),

        "tunnel_used_violation_when_loaded":
            int(rep["tunnel_used_violation_when_loaded"].sum()),

        "dist_asym_gt_0_2":
            int(rep["dist_asym_gt_0_2"].sum()),

        "dist_violate_short_fast":
            int(rep["dist_violate_short_fast"].fillna(False).sum()),

        "time_violate_fast_vs_short":
            int(rep["time_violate_fast_vs_short"].fillna(False).sum()),

        "safe_dominates_fastest":
            int(rep["safe_dominates_fastest"].fillna(False).sum()),

        "all_profiles_identical":
            int(rep["all_profiles_identical"].fillna(False).sum()),

        "dist_outlier_iqr":
            int(rep["dist_outlier_iqr"].sum()),

        "time_outlier_iqr":
            int(rep["time_outlier_iqr"].sum()),
    }

    summary_df = pd.DataFrame([summary])

    return rep, summary_df


if __name__ == "__main__":

    od_csv_path = (
        r"C:\Users\9631\OneDrive - Stiebel Eltron GmbH & Co. KG\Desktop\OR\od_matrix_large.csv"
    )

    try:
        _, summary = run_checks(
            od_csv=od_csv_path
        )

    except Exception as e:
        print(f"[FEHLER] {e}")

    else:
        print("\nOD-Matrix-Validierung abgeschlossen.\n")
        print(summary.to_string(index=False))


OD-Matrix-Validierung abgeschlossen.

 rows_found  rows_expected_theoretical_estimate  diag_violations  negative_dist  negative_time  negative_cost  cost_inf  cost_nan  min_per_km_outside_gross_band  pathlen_violations  v_avg_outside_bounds  tunnel_used_violation_when_loaded  dist_asym_gt_0_2  dist_violate_short_fast  time_violate_fast_vs_short  safe_dominates_fastest  all_profiles_identical  dist_outlier_iqr  time_outlier_iqr
      15300                               15300                0              0              0              0         0         0                              0                   0                     0                                  0               262                        0                           0                       0                     450                49                65
